# B5T1 part B in a notebook: eval a prompt, let GEPA rewrite it, eval it again

This notebook runs the steps of [README.md](README.md) one cell at a time. The README is the full guide, with the
same steps as terminal commands: in each step, the cells below the step's text replace those commands. Run the
cells from the top, in order.

Each step starts with a prediction. Write it down in your notes before you run the step's cells, then open the
step's **Check your predictions** box at the end of the step.

The cells call the kit's own files: `check_gateway.py`, `task.py` (with Inspect), `optimize.py`, `budget.py` and
`compare.py`. Open any of them to see what a cell does.

## Set up

Select the kernel of this folder's `.venv` (VS Code: **Select Kernel** › **Python Environments** › `.venv`).
If you started Jupyter from a terminal where `KB_MCP_TOKEN` is set, the first cell does not ask for anything.
Otherwise it asks for your token in a hidden box. Never type the token into a cell's code: a saved notebook keeps
its cells.

In [ ]:
import getpass
import os

if not os.environ.get("KB_MCP_TOKEN"):
    os.environ["KB_MCP_TOKEN"] = getpass.getpass("Paste your course token, then press Enter: ")
print("KB_MCP_TOKEN is set" if os.environ.get("KB_MCP_TOKEN") else "KB_MCP_TOKEN is NOT set")

You should see `KB_MCP_TOKEN is set`.

In [ ]:
import check_gateway

problem = check_gateway.main()

You should see ✓ lines and `OK: the gateway works with your token.` A line starting `PROBLEM:` says what to fix.

## Step 1: run the starting prompt

```text
prompt.txt + held-out test cases -> worker model -> SQL -> code scorer
```

Inspect runs the worker model once per case with `prompt.txt` as its system prompt. The code scorer runs the
SQL and compares its number with the reference.

**Predict:** Write down whether you expect the starting prompt to pass below half, about half, or more than half
of the held-out cases. What kind of question do you think might fail?

In [ ]:
print(open("prompt.txt", encoding="utf-8").read())

You should see the starting prompt: the table and its columns, and the instruction to write one query.

In [ ]:
import inspect_ai

from task import sql_cases

before_logs = inspect_ai.eval(sql_cases(split="test", prompt="prompt.txt"), log_dir="logs", display="none")
before = before_logs[0]
print("eval finished:", before.status, "-", len(before.samples), "held-out cases")

The eval takes under a minute. You should see `eval finished: success - 12 held-out cases`. The cell does not print the score yet.

<details>
<summary>Check your predictions</summary>

The score is the fraction of held-out cases that pass. Each case contributes the same share. In L15, an Inspect
eval has a dataset, a solver and a scorer: here the cases supply inputs and targets, the worker supplies SQL, and
code checks the result.
</details>

## Step 2: read and name the failures

```text
held-out run -> failed cases -> failure names -> score
```

Read the cases before the score so the aggregate does not direct your error analysis. A failure name is short
and says what to fix. Invent each name after reading a failed case, and reuse it when another case fails in the
same way.

**Predict:** Write down whether you expect several failed cases to share a failure name, or each failure to need
a different name.

In [ ]:
for sample in before.samples:
    check = sample.scores["sql_check"]
    if check.value == 0:
        print("case", sample.id)
        print(check.explanation)
        print("-" * 70)

You should see one block per failed case: the question, the reference SQL and value, and `FAILURE:` with what the checker found and the model's SQL.

**Your error-analysis table.** Double-click this cell to edit it. One row per failed case, in your own words.

| case id | what went wrong | name of the failure |
|---|---|---|
| | | |
| | | |
| | | |

In [ ]:
for score in before.results.scores:
    if score.name == "sql_check":
        print("held-out score, starting prompt:", round(score.metrics["accuracy"].value, 3))

You should see the score: the share of the held-out cases that passed, between 0 and 1.

<details>
<summary>Check your predictions</summary>

Cases with the same failure should have the same name, and the counts across your names should add up to the
number of failed cases. Reading the failures before the score is error analysis: the cases tell you what to change,
and the score only tells you how many cases need it.
</details>

## Step 3: let GEPA rewrite the prompt

```text
train cases -> failures -> reflection model -> prompt candidates -> val cases -> selected prompt
```

GEPA uses train cases and their feedback to propose and test prompt changes. It keeps a Pareto set of candidates
that each do best on some examples and returns a prompt selected using val scores; GEPA does not read the test
cases.

**Predict:** Write down whether you expect the new prompt to address at least one of your failure names. Do you
expect GEPA to consult the held-out cases?

In [ ]:
import optimize

stopped = optimize.main([])

You should see the estimate first, then lines like `Iteration 1: Proposed new text for predict:` with each new prompt, and at the end the val scores, the cost in CAD and `wrote prompt.optimized.txt`. It takes about a minute, longer when the gateway is busy. If the cost meter stops GEPA, it says so and still writes the best prompt it had.

<details>
<summary>Check your predictions</summary>

GEPA uses train and val cases. It does not read any held-out test case. L17's evaluator-optimizer
pattern uses feedback to propose revisions and a limit to stop attempts. A prompt change can address a failure
without guaranteeing a better held-out score.
</details>

## Step 4: read the new prompt

```text
prompt.txt + prompt.optimized.txt -> added lines -> failure names and data check
```

A prompt can improve an eval and still contain a rule that is not true of the data. Read the added lines before
using the prompt and check their claims.

**Predict:** Write down whether you expect every failure name in your table to be addressed, and whether you
expect every added rule to be true of the data.

In [ ]:
import difflib

old = open("prompt.txt", encoding="utf-8").read().splitlines()
new = open("prompt.optimized.txt", encoding="utf-8").read().splitlines()
for line in difflib.unified_diff(old, new, "prompt.txt", "prompt.optimized.txt", lineterm=""):
    print(line)

You should see two file headers starting `---` and `+++`, then the prompt lines GEPA removed (starting `-`) and added (starting `+`).

<details>
<summary>Check your predictions</summary>

An added line can address a failure name, leave a name unaddressed, or make a claim you cannot verify from the
data script. L17's evaluator-optimizer pattern checks a result against criteria. A reader still needs to check
claims before using the prompt.
</details>

## Step 5: eval the new prompt

```text
prompt.optimized.txt + same held-out test cases -> worker model -> code scorer
```

Run the same eval on the same test split so the scores and individual cases can be compared. GEPA selected a
prompt using val cases, so the held-out cases provide a separate check.

**Predict:** Write down whether you expect the held-out score to be higher, lower, or about the same, and whether
you expect the same or fewer failures.

In [ ]:
after_logs = inspect_ai.eval(sql_cases(split="test", prompt="prompt.optimized.txt"), log_dir="logs",
                            display="none")
after = after_logs[0]
print("eval finished:", after.status, "-", len(after.samples), "held-out cases")
still_failing = 0
for sample in after.samples:
    check = sample.scores["sql_check"]
    if check.value == 0:
        print("case", sample.id)
        print(check.explanation)
        print("-" * 70)
        still_failing = still_failing + 1
print("cases that still fail:", still_failing)
for score in after.results.scores:
    if score.name == "sql_check":
        print("held-out score, new prompt:", round(score.metrics["accuracy"].value, 3))

You should see `eval finished: success - 12 held-out cases` and one block for each case that still fails, if any. The last two lines give the number of cases that still fail and the new prompt's score on the held-out cases.

<details>
<summary>Check your predictions</summary>

Compare whether the after score is at least the before score and whether the new prompt has the same or fewer
failures. Either relation can fail on a particular run. L17 uses held-out cases because GEPA chose the prompt on
validation cases, so the held-out score checks questions outside that choice.
</details>

## Step 6: price the session

```text
upper-bound estimate -> eval and GEPA records -> actual cost comparison
```

Your token's term budget is 2 CAD. The course rule for one whole run (check, eval, GEPA, eval again) is less than
0.10 CAD. `budget.py` prints an upper-bound estimate of the calls and their cost before any call is made, and
`optimize.py` printed the same kind of estimate at the start of Step 3. `compare.py` reads the records in `logs/`
and reports what the calls actually cost.

**Predict:** Write down whether you expect the recorded total to be above or below the
upper-bound estimate.
Do you expect a few cents or less?

In [ ]:
import budget

estimate = budget.print_estimate()

You should see the estimate: the calls and the most each part of the run can cost, before any call.

In [ ]:
import compare

missing = compare.main()

You should see one line per eval run with its score and cost in CAD, the cost of GEPA, the total, and the check line on the held-out cases.

<details>
<summary>Check your predictions</summary>

The recorded total should be below the upper-bound estimate for the corresponding run. L17's course LLM gateway
reserves the most a call can cost before the call and replaces the reserve with the actual cost after the reply;
bounded attempts help keep spending within a limit.
</details>

## Next

Go on with **4. Extra exercises** and **6. Recap and reflection** in [README.md](README.md).